# Notebook 06 — Post-Translational Modifications: Survey

**Module:** 07 — Biochemistry and Molecular Biology  
**Notebook:** 06 of 10  
**Prerequisites:** NB02 (amino acids), NB05 (translation)  
**Time estimate:** 60 minutes

> **Tier 2 scope:** Survey the major PTM types, understand their functional consequences,
> and implement a phosphorylation cascade simulation. Full structural biochemistry of
> each PTM is Module 11 territory.

---
## Step 1 — Motivation

The proteome is not just the set of translated sequences — it is the set of all
modified forms those sequences can take. A single protein can exist in hundreds of
functional states depending on which residues are phosphorylated, ubiquitinated,
or acetylated at any given moment. Phosphoproteomics (Module 10) generates data
specifically to measure these states. Understanding PTMs is required to interpret
that data.

---
## Step 3 — Biological Background

### Major PTM types

| PTM | Residues modified | Effect | Enzyme adds | Enzyme removes |
|-----|------------------|--------|------------|---------------|
| **Phosphorylation** | Ser, Thr, Tyr | Activates/inactivates; creates docking sites | Kinase | Phosphatase |
| **Ubiquitination** | Lys | Marks for proteasomal degradation (mono: signalling) | E1/E2/E3 cascade | Deubiquitinase (DUB) |
| **Acetylation** | Lys, N-terminus | Neutralises positive charge; histone code; protein stability | HAT (histones) / KAT | HDAC / KDAC |
| **Methylation** | Lys, Arg | Activating or repressing (histone H3K4me3 = active; H3K27me3 = repressive) | HMT | Demethylase |
| **SUMOylation** | Lys | Nuclear functions, DNA repair, stress response | SUMO E1/E2/E3 | SENP protease |
| **Glycosylation** | Ser/Thr (O-) or Asn (N-) | Protein folding, trafficking, cell-surface recognition | Glycosyltransferases | Glycosidases |
| **Palmitoylation** | Cys | Membrane anchoring | DHHC enzymes | APT thioesterases |

### Phosphorylation cascades

Signal → Receptor tyrosine kinase → MAPK cascade (Ras → Raf → MEK → ERK).
Each kinase phosphorylates and activates the next. The cascade amplifies the signal:
one ligand → thousands of activated ERK molecules. This is why kinase inhibitors
(imatinib, vemurafenib) are effective cancer drugs.

### Databases for PTM research

- **PhosphoSitePlus** (phosphosite.org): >800,000 phosphorylation sites, kinase
  predictions, disease-linked PTMs
- **UniMod** (unimod.org): standard PTM mass registry for mass spectrometry searches
- **STRING** (string-db.org): protein interaction networks including kinase-substrate links

---
## Step 4 — Mathematical Explanation

**PTM mass shift (important for mass spec):**
Each PTM adds a specific mass to the modified peptide:

| PTM | Mass shift (Da) |
|-----|----------------|
| Phosphorylation (pSer, pThr, pTyr) | +79.966 |
| Ubiquitin remnant (Lys-GG) | +114.043 |
| Acetylation | +42.011 |
| Monomethylation | +14.016 |
| Dimethylation | +28.031 |
| Trimethylation | +42.047 |

**Phosphorylation cascade dynamics:**
A simple 3-step kinase cascade can be modelled as:
$$\frac{d[A^*]}{dt} = k_{act} \cdot [S] \cdot [A] - k_{deact} \cdot [A^*]$$
where [S] = upstream signal, [A] = inactive kinase, [A*] = active kinase,
k_act = activation rate, k_deact = deactivation/phosphatase rate.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
# Cell 6.1 — PTM mass shift calculator
PTM_MASSES = {
    'phosphorylation': 79.966,
    'ubiquitin_remnant': 114.043,
    'acetylation': 42.011,
    'monomethylation': 14.016,
    'dimethylation': 28.031,
    'trimethylation': 42.047,
    'sumoylation_remnant': 2530.0,   # full SUMO2 tag
}

def peptide_mass_with_ptms(base_mass: float, ptms: list) -> float:
    """
    Calculate observed peptide mass given base mass and list of PTMs.
    ptms: list of PTM names from PTM_MASSES
    """
    total_shift = sum(PTM_MASSES.get(p, 0) for p in ptms)
    return base_mass + total_shift

# Example: EGFR peptide with Y1068 phosphorylation
egfr_peptide_mass = 1412.68  # unmodified mass (Da)
ptms_observed = ['phosphorylation']
obs_mass = peptide_mass_with_ptms(egfr_peptide_mass, ptms_observed)
print(f"EGFR Y1068 peptide:")
print(f"  Unmodified mass:    {egfr_peptide_mass:.3f} Da")
print(f"  With phospho-Y:     {obs_mass:.3f} Da (+{PTM_MASSES['phosphorylation']:.3f})")
print()

# Histone H3K27: trimethylated vs. acetylated
h3k27_base = 836.47
print("Histone H3K27 modifications (same lysine, different marks):")
for ptm in ['trimethylation', 'acetylation']:
    m = peptide_mass_with_ptms(h3k27_base, [ptm])
    print(f"  K27{ptm[:2].upper()}: {m:.3f} Da")
print("  (These require different database mass tolerances to distinguish)")

In [ ]:
# Cell 6.2 — Phosphorylation cascade ODE simulation (3-step MAPK)
def mapk_cascade_ode(t, y, k_ras_on, k_ras_off, k_raf_on, k_raf_off,
                      k_mek_on, k_mek_off, k_erk_on, k_erk_off,
                      signal_amplitude):
    """
    Simplified 4-component MAPK cascade:
    y[0] = Ras-GTP (active Ras), y[1] = pRaf, y[2] = ppMEK, y[3] = ppERK

    Signal drives Ras activation; each component activates the next.
    Phosphatases provide deactivation.
    """
    ras_total = 1.0  # normalised total concentrations
    signal = signal_amplitude  # constant external signal

    ras_active, raf_active, mek_active, erk_active = y
    ras_inactive = ras_total - ras_active

    d_ras = k_ras_on * signal * ras_inactive - k_ras_off * ras_active
    d_raf = k_raf_on * ras_active * (1 - raf_active) - k_raf_off * raf_active
    d_mek = k_mek_on * raf_active * (1 - mek_active) - k_mek_off * mek_active
    d_erk = k_erk_on * mek_active * (1 - erk_active) - k_erk_off * erk_active

    return [d_ras, d_raf, d_mek, d_erk]


# Solve with step-change signal at t=5
t_span = (0, 60)
t_eval = np.linspace(0, 60, 600)
y0 = [0.0, 0.0, 0.0, 0.0]  # all inactive initially

params = dict(k_ras_on=2.0, k_ras_off=0.5, k_raf_on=3.0, k_raf_off=0.8,
              k_mek_on=4.0, k_mek_off=0.5, k_erk_on=5.0, k_erk_off=0.3,
              signal_amplitude=1.0)

sol = solve_ivp(mapk_cascade_ode, t_span, y0, t_eval=t_eval,
                args=tuple(params.values()), method='RK45')

print("MAPK cascade steady-state levels:")
for name, vals in zip(['Ras-GTP', 'pRaf', 'ppMEK', 'ppERK'], sol.y):
    print(f"  {name}: {vals[-1]:.3f} (fraction of total)")
print("  [ERK = effector; nearly fully activated at steady state]")

In [ ]:
# Cell 6.3 — Ubiquitin-proteasome: simulate protein degradation
def protein_degradation_model(t_max: float, k_synthesis: float,
                               k_ubiquitination: float, k_degradation: float,
                               stimulus_time: float = 10.0,
                               stimulus_strength: float = 5.0,
                               n_points: int = 500) -> tuple:
    """
    Model protein abundance over time with ubiquitin-proteasome degradation.
    A stimulus at stimulus_time increases ubiquitination rate.
    """
    t = np.linspace(0, t_max, n_points)
    dt = t[1] - t[0]

    protein = np.zeros(n_points)
    ubiquitinated = np.zeros(n_points)
    protein[0] = k_synthesis / (k_ubiquitination + 0.01)  # steady state

    for i in range(1, n_points):
        # Ubiquitination rate spikes after stimulus
        k_ub_effective = k_ubiquitination * (1 + stimulus_strength * (t[i] > stimulus_time))

        dp = k_synthesis - k_ub_effective * protein[i-1]
        dub = k_ub_effective * protein[i-1] - k_degradation * ubiquitinated[i-1]

        protein[i] = max(0, protein[i-1] + dp * dt)
        ubiquitinated[i] = max(0, ubiquitinated[i-1] + dub * dt)

    return t, protein, ubiquitinated

t_deg, prot, ubiq = protein_degradation_model(
    t_max=60, k_synthesis=1.0, k_ubiquitination=0.05,
    k_degradation=0.5, stimulus_time=15.0, stimulus_strength=5.0
)
print(f"Protein levels: before stimulus={prot[:200].mean():.2f}, "
      f"after stimulus={prot[300:].mean():.2f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — MAPK cascade dynamics + protein degradation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: MAPK cascade activation
ax = axes[0]
components = ['Ras-GTP', 'pRaf', 'ppMEK', 'ppERK']
colors_map = ['tomato', 'orange', 'steelblue', 'seagreen']
for comp, vals, color in zip(components, sol.y, colors_map):
    ax.plot(sol.t, vals, color=color, lw=2, label=comp)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Active fraction (0–1)')
ax.set_title('MAPK cascade activation:\nsignal amplification through phosphorylation')
ax.legend(fontsize=8)
ax.set_ylim(-0.02, 1.05)

# Panel 2: Protein degradation via ubiquitin-proteasome
ax = axes[1]
ax.plot(t_deg, prot, color='steelblue', lw=2, label='Unmodified protein')
ax.plot(t_deg, ubiq, color='tomato', lw=2, ls='--', label='Ubiquitinated protein')
ax.axvline(15.0, color='gray', ls=':', lw=1.5, label='Ubiquitination signal (t=15)')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Protein abundance (a.u.)')
ax.set_title('Ubiquitin-proteasome degradation:\nstimulus triggers rapid protein clearance')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. A mass spec experiment detects a peptide with observed mass = unmodified + 79.97 Da.
   Which PTM is this? Which residues could carry it? How would you distinguish
   pSer from pThr from pTyr experimentally?
2. Vemurafenib is a BRAF kinase inhibitor used in melanoma. Using the MAPK cascade
   model, simulate what happens if k_raf_on is set to near-zero. Does ERK activation
   drop to zero? Why or why not?
3. H3K4me3 marks active promoters; H3K27me3 marks repressed genes. Both are
   trimethylations on histone H3 lysines, yet have opposite effects. What determines
   the functional outcome of a methylation mark?
4. The ubiquitin-proteasome system (UPS) degrades short-lived regulatory proteins.
   PROTAC drugs hijack this system to degrade otherwise-undruggable targets. Explain
   mechanistically how a PROTAC molecule causes protein degradation.

---
## Quiz — Active Recall

1. Name three PTMs that modify lysine. How do their effects differ?
2. What is the mass shift of phosphorylation? Why does this matter for mass spectrometry?
3. What is the difference between a kinase and a phosphatase?
4. Why are lysine residues particularly important for ubiquitination?
5. What database would you use to look up known phosphorylation sites on EGFR?

---
## Reflection

**Date completed:** ____________________

> *[A phosphoproteomics experiment finds 500 significantly upregulated phosphosites
> after EGF treatment. How would you begin to interpret which phosphosites are
> functionally important vs. bystander events?]*

---
*Next: `07_biochemistry_assay_data_shapes.ipynb`*